# IEEE fraud model training

Reads the processed train, validation, and test parquet files and trains an XGBoost model. The split already happened in `01_data_transformation.ipynb`.

Tracking goes to the Docker MLflow server (`http://localhost:5000`). That server proxies artifacts into the MinIO bucket `s3://mlflow/` (`http://localhost:9000`).

In [1]:
from pathlib import Path
import os

import mlflow
import pandas as pd

from src.run_time_configuration import BaseConfigParams
from src.model_training.training import ModelTrainingWorkflow

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "runs_entrypoints":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TARGET_COLUMN = "is_fraud"
SEED = 42
TOP_N_FEATURES = 40
DROP_FROM_FEATURES = ["transaction_id", "seconds_from_reference", TARGET_COLUMN]


In [2]:

def _load_env(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}")
    for raw in path.read_text().splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip())


_load_env(PROJECT_ROOT / ".env")

MLFLOW_TRACKING_URI = "http://localhost:5000"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed dir: {PROCESSED_DIR}")
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")

Project root: /home/martinezyamamotoarthur/modeling_fraud_system
Processed dir: /home/martinezyamamotoarthur/modeling_fraud_system/data/processed
MLflow tracking URI: http://localhost:5000


In [3]:
df_train = pd.read_parquet(PROCESSED_DIR / "train.parquet")
df_validation = pd.read_parquet(PROCESSED_DIR / "validation.parquet")
df_test = pd.read_parquet(PROCESSED_DIR / "test.parquet")

for name, frame in {
    "train": df_train,
    "validation": df_validation,
    "test": df_test,
}.items():
    print(name, frame.shape, "pos_rate", round(float(frame[TARGET_COLUMN].mean()), 4))

train (331633, 332) pos_rate 0.0352
validation (82909, 332) pos_rate 0.0352
test (175998, 332) pos_rate 0.0344


In [4]:
def features_and_target(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    y = df[TARGET_COLUMN]
    x = df.drop(columns=[column for column in DROP_FROM_FEATURES if column in df.columns])
    return x, y


runtime_params = BaseConfigParams(
    experiment_name="ieee_fraud",
    mlflow_run_name="ieee_xgb",
    training_start_date="2017-12-01",
    training_end_date="2018-03-31",
    test_start_date="2018-04-01",
    test_end_date="2018-06-30",
)

workflow = ModelTrainingWorkflow(runtime_params)
workflow.df_train = df_train
workflow.df_validation = df_validation
workflow.df_test = df_test
workflow.x_train, workflow.y_train = features_and_target(df_train)
workflow.x_validation, workflow.y_validation = features_and_target(df_validation)
workflow.x_test, workflow.y_test = features_and_target(df_test)
workflow.generate_preprocessed_data(numeric_only=True)

print("train", workflow.x_train_preproc.shape, "pos_rate", float(workflow.y_train.mean()))
print("valid", workflow.x_validation_preproc.shape, "pos_rate", float(workflow.y_validation.mean()))
print("test ", workflow.x_test_preproc.shape, "pos_rate", float(workflow.y_test.mean()))
print("numeric features", len(workflow.x_train_preproc.columns))

2026-09-27 12:45:21,477 - src.model_training.training - INFO - Initialized the SolvencyNewCustomersModelTraining workflow.
2026-09-27 12:45:21,487 - src.model_training.training - INFO - Generating preprocessed data by copying original data.
2026-09-27 12:45:22,581 - src.model_training.training - INFO - Filtering to numeric columns: 272 out of 329 total columns
2026-09-27 12:45:22,582 - src.model_training.training - INFO - Selected numeric columns: ['amount_usd', 'card1', 'card2', 'card_issue_country', 'card5', 'billing_region', 'billing_country', 'dist1', 'dist2', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'transaction_time_of_day', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'V1', 'V2', 'V4', 'V6', 'V8', 'V10', 'V12', 'V14', 'V15', 'V19', 'V23', 'V25', 'V27', 'V29', 'V35', 'V37', 'V39', 'V41', 'V44', 'V46', 'V48', 'V52', 'V53', 'V55', 'V56', 'V57', 'V61', 'V65', 'V66', 'V68', 'V69', 'V75', 'V77

train (331633, 272) pos_rate 0.035219655462514286
valid (82909, 272) pos_rate 0.035219336863307964
test  (175998, 272) pos_rate 0.03444925510517165
numeric features 272


In [5]:
results = workflow.train_model(
    hyperparameters={
        "n_estimators": 200,
        "max_depth": 5,
        "learning_rate": 0.03,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    },
    model_type="xgboost",
    log_into_mlflow=True,
    show_plots=False,
    verbose=False,
    seed=SEED,
    run_name="ieee_xgb_01",
    top_n_features=TOP_N_FEATURES,
)

metrics_table = pd.DataFrame(
    {
        "train": results["train_metrics"],
        "validation": results["val_metrics"],
        "test": results["test_metrics"],
    }
).loc[["accuracy", "precision", "recall", "f1", "auc", "pr_auc", "ks_statistic"]]

run_id = workflow.run_time_config.mlflow_run_id
client = mlflow.tracking.MlflowClient()
run = client.get_run(run_id)

print("threshold", round(results["threshold"], 4))
print("mlflow run id", run_id)
print("tracking uri", mlflow.get_tracking_uri())
print("artifact uri", run.info.artifact_uri)
print("logged artifacts", [a.path for a in client.list_artifacts(run_id)])
display(results["top_features"])
metrics_table.round(4)

2026-09-27 12:45:23,235 - src.model_training.training - INFO - Starting model training without hyperparameter tuning using xgboost.
2026-09-27 12:45:23,466 - src.model_training.training - INFO - Training xgboost model with fixed hyperparameters.
/home/martinezyamamotoarthur/modeling_fraud_system/.venv/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [12:45:28] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
2026-09-27 12:45:51,187 - src.model_training.training - INFO - Calculated optimal threshold (F1-based) on validation set: 0.2301
2026-09-27 12:45:51,924 - src.model_training.evaluation - INFO - Training Metrics - AUC: 0.9407, PR-AUC: 0.6939, KS: 0.7315
2026-09-27 12:45:51,990 - src.model_training.evaluation - INFO - Validation Metrics - AUC: 0.9364, PR-AUC: 0.6789, KS: 0.7229
2026-09-27 12:45:52,123 - src.model_training.evaluation - INFO - Test Metrics - AUC: 0.8845, P

🏃 View run ieee_xgb_01 at: http://localhost:5000/#/experiments/52/runs/262ff230c45e4cba819e165a782ee8db
🧪 View experiment at: http://localhost:5000/#/experiments/52
threshold 0.2301
mlflow run id 262ff230c45e4cba819e165a782ee8db
tracking uri http://localhost:5000
artifact uri mlflow-artifacts:/52/262ff230c45e4cba819e165a782ee8db/artifacts
logged artifacts ['feature_importance', 'plots']


,rank,feature,importance
0,1,V258,888.644897
1,2,card1_x_product_channel_target_rate,657.890320
2,3,V188,556.934814
3,4,V242,494.062531
4,5,V187,421.541046
5,6,card1_target_rate,418.357727
6,7,V283,352.895508
7,8,card_V309_nunique,325.563873
8,9,V95,300.795563
9,10,V148,290.301514


,train,validation,test
accuracy,0.977541,0.976649,0.970329
precision,0.714692,0.699352,0.597406
recall,0.603082,0.591096,0.425367
f1,0.65416,0.640683,0.496917
auc,0.940743,0.936411,0.884492
pr_auc,0.693909,0.67888,0.504087
ks_statistic,0.731451,0.72295,0.612679
